In [1]:
#r "C:\Users\user\Desktop\Ainur\practice2026\task17\bin\Debug\net10.0\task17.dll"
#r "nuget:ScottPlot,5.0.54"

using System;
using System.Collections.Generic;
using System.Threading;
using ScottPlot;
using task17;

public class TestCommand : ICommand, IRepeatable
{
    private readonly int _id;
    private int _counter;
    private readonly int _maxCalls;
    
    public int Id => _id;
    public int Counter => _counter;

    public TestCommand(int id, int maxCalls)
    {
        _id = id;
        _maxCalls = maxCalls;
    }

    public void Execute()
    {
        _counter++;
        Console.WriteLine($"Поток {_id} вызов {_counter}");
    }

    public bool IsFinished()
    {
        return _counter >= _maxCalls;
    }
}

Console.WriteLine("=== Демонстрация длительных операций ===\n");

var scheduler = new RoundRobinScheduler();
var commands = new TestCommand[5];

for (int i = 0; i < 5; i++)
{
    commands[i] = new TestCommand(i + 1, 3);
    scheduler.Add(commands[i]);
}

var timeline = new List<(int step, int cmd1, int cmd2, int cmd3, int cmd4, int cmd5)>();
int step = 0;

Console.WriteLine("Начало выполнения:\n");

while (scheduler.HasCommand())
{
    step++;
    var cmd = scheduler.Select();
    cmd.Execute();

    if (cmd is IRepeatable r && !r.IsFinished())
        scheduler.Add(cmd);
    
    timeline.Add((step, commands[0].Counter, commands[1].Counter, commands[2].Counter, commands[3].Counter, commands[4].Counter));
}

Console.WriteLine("\nВсе команды выполнены по 3 раза");

var plt = new Plot();
plt.Title("Выполнение длительных операций (Round Robin)");
plt.XLabel("Шаг");
plt.YLabel("Количество вызовов");

var steps = new double[timeline.Count];
for (int i = 0; i < timeline.Count; i++)
    steps[i] = i + 1;

var colors = new[] { "#FF0000", "#00FF00", "#0000FF", "#FFA500", "#800080" };

for (int i = 0; i < 5; i++)
{
    var values = new double[timeline.Count];
    for (int j = 0; j < timeline.Count; j++)
    {
        values[j] = i switch
        {
            0 => timeline[j].cmd1,
            1 => timeline[j].cmd2,
            2 => timeline[j].cmd3,
            3 => timeline[j].cmd4,
            4 => timeline[j].cmd5,
            _ => 0
        };
    }
    
    var line = plt.Add.Scatter(steps, values);
    line.LegendText = $"Поток {i + 1}";
    line.LineWidth = 2;
    line.MarkerSize = 6;
    line.Color = ScottPlot.Color.FromHex(colors[i]);
}

plt.ShowLegend();
plt.SavePng("long_operations.png", 800, 600);
Console.WriteLine("\nГрафик: long_operations.png");

Installed Packages ScottPlot, 5.0.54

Loading extensions from `C:\Users\user\.nuget\packages\skiasharp\2.88.9\interactive-extensions\dotnet\SkiaSharp.DotNet.Interactive.dll`

=== Демонстрация длительных операций ===

Начало выполнения:

Поток 1 вызов 1
Поток 2 вызов 1
Поток 3 вызов 1
Поток 4 вызов 1
Поток 5 вызов 1
Поток 1 вызов 2
Поток 2 вызов 2
Поток 3 вызов 2
Поток 4 вызов 2
Поток 5 вызов 2
Поток 1 вызов 3
Поток 2 вызов 3
Поток 3 вызов 3
Поток 4 вызов 3
Поток 5 вызов 3

Все команды выполнены по 3 раза

График: long_operations.png
